### Algorithms for Massive Datasets Project Source Code
- Author: Andrea Colombo

In [1]:
# Kaggle setup for downloading the dataset, this has to be handled before
# running the rest of the notebook to avoid errors
import os

os.environ["KAGGLE_USERNAME"] = "x"
os.environ["KAGGLE_KEY"] = "x"

In [ ]:
# Generic configuration related to the actual project implementation
# The user can change how much of the dataset is actually used during the run
# and other global variables that influence the behaviour of the notebook

ENABLE_LOGGING = True
SAMPLING_PROPORTION = 0.5 # This can range from 0.0 (excluded) to 1.0
assert 0 < SAMPLING_PROPORTION <= 1
RAND_SEED = 42 # Set as None for random, used for dataset sampling
FM_NUM_HASHES = 2 # Number of hash function used for the FM algorithm
FM_PHI = 0.77351

def log(s):
    if ENABLE_LOGGING == True:
        print(f"[LOG] {s}")


In [ ]:
# Get the dataset using kaggle API (please check the kaggle auth cell before running
# this one)

%pip install kaggle
%pip install xxhash
!kaggle datasets download -d "benjaminawd/new-york-times-articles-comments-2020"
!unzip -d dataset new-york-times-articles-comments-2020.zip && rm -r new-york-times-articles-comments-2020.zip
# !ls -l dataset

Dataset URL: https://www.kaggle.com/datasets/benjaminawd/new-york-times-articles-comments-2020
License(s): CC-BY-NC-SA-4.0
new-york-times-articles-comments-2020.zip: Skipping, found more recently modified local copy (use --force to force download)
Archive:  new-york-times-articles-comments-2020.zip
replace dataset/nyt-articles-2020.csv? [y]es, [n]o, [A]ll, [N]one, [r]ename: total 5037592
-rw-r--r-- 1 root root    7942395 Jul 20  2021 nyt-articles-2020.csv
-rw-r--r-- 1 root root 3066945799 Jul 20  2021 nyt-comments-2020.csv
-rw-r--r-- 1 root root  303290997 Jul 20  2021 nyt-comments-part0.csv
-rw-r--r-- 1 root root  303524505 Jul 20  2021 nyt-comments-part1.csv
-rw-r--r-- 1 root root  302002043 Jul 20  2021 nyt-comments-part2.csv
-rw-r--r-- 1 root root  309656964 Jul 20  2021 nyt-comments-part3.csv
-rw-r--r-- 1 root root  309020152 Jul 20  2021 nyt-comments-part4.csv
-rw-r--r-- 1 root root  309869892 Jul 20  2021 nyt-comments-part5.csv
-rw-r--r-- 1 root root  246218752 Aug 31 16:40 nyt-

In [4]:
%pip install pyspark
%pip install py4j

import pyspark


spark_session = pyspark.sql.SparkSession.builder.getOrCreate()
spark_contenxt = spark_session.sparkContext

In [33]:
opts = {
    "header": "true",
    "inferSchema": "true",
    "quote": '"',
    "escape": '"',
    "multiLine": "true",
    "mode": "PERMISSIVE"
}

df = spark_session.read.options(
    **opts
).csv(
    "dataset/nyt-comments-2020.csv",
    header=True
).sample(
    withReplacement=False,
    fraction=SAMPLING_PROPORTION,
    seed=RAND_SEED
).cache()

# df.show(5, truncate=False)
# df.printSchema() 
log(f"Dataset columns {df.columns}")
log(f"Dataset count {df.count()}")

[LOG] Dataset columns ['commentID', 'status', 'commentSequence', 'userID', 'userDisplayName', 'userLocation', 'userTitle', 'commentBody', 'createDate', 'updateDate', 'approveDate', 'recommendations', 'replyCount', 'editorsSelection', 'parentID', 'parentUserDisplayName', 'depth', 'commentType', 'trusted', 'recommendedFlag', 'permID', 'isAnonymous', 'articleID']
[LOG] Dataset count 2495514


In [26]:
# Set up user streams for partitions
# The dataset has null entries so they have to be cleaned up to avoid
# null strings giving us garbage with the hashes
users_stream = (
    df.
    select("userID").
    where("userID IS NOT NULL").
    rdd.
    map(lambda row: row["userID"])
)

# help(users_stream)

# FM Implementation

Each userID is used to calculate _n_ hash functions (FM_NUM_HASHES). We keep track of the
maximum number of trailing zeros for each has function and we use the registers of each
partition for the final estimate

In [27]:
import xxhash
import statistics

def ctz(x):
    if x == 0:
        return 64
    return (x & -x).bit_length() - 1

def hash64bits(value, seed):
    return xxhash.xxh64(value.encode("utf-8"), seed=seed).intdigest()

def fm_partition(it):
    registers = [0] * FM_NUM_HASHES
    for user in it:
        for i in range(FM_NUM_HASHES):
            bits = hash64bits(user, i)
            r = ctz(bits)
            if r > registers[i]:
                registers[i] = r

    yield registers


In [28]:
locsketches = users_stream.mapPartitions(fm_partition)

reduced_registers = locsketches.reduce(
    lambda a, b: [max(x, y) for x, y in zip(a, b)]
)

estimates = [(2 ** r) / FM_PHI for r in reduced_registers]
fm_estimate = statistics.median(estimates)

print("FM estimate:", round(fm_estimate))

FM estimate: 677804


In [29]:
# Check exact answer for reference
# Average running time in the development runs was about 55s (full dataset).
df.select("userID").distinct().count()

1468406

In [30]:
users_stream.count()

3800675